# Diseño Experimental: Detección Binaria de Caídas (CNN-BiLSTM) — GPU

**Objetivo:** Comparar el rendimiento de un modelo CNN-BiLSTM en detección binaria Fall/ADL
entrenado sobre dos conjuntos unificados a 50 Hz, con validación cruzada estratificada
por sujeto y métricas agregadas (Sensibilidad / Especificidad / Precisión).

### Requisitos (GPU)

- **TensorFlow con soporte CUDA** (Linux nativo o WSL2). En Windows nativo TF ≥ 2.11
  ya no incluye GPU — usar `tensorflow-directml` o ejecutar dentro de WSL2. Ver
  `https://www.tensorflow.org/install/pip`.
- **Drivers NVIDIA actualizados** y CUDA/cuDNN compatibles con la versión de TF instalada.
- RAM de GPU ≥ 4 GB recomendada (batch=512 con secuencia (150, 8) ocupa ~1.2 GB).

### Auto-detección

La celda 1 imprime las GPUs detectadas y activa `mixed_float16` (float16 en capas
convolucionales/recurrentes, float32 en la salida del modelo). Si no hay GPU, sigue
ejecutable en CPU pero con los hiperparámetros ajustados al dataset completo el
entrenamiento será muy lento.

### Contrato de Datos de Entrada
Capa `oro/falls/` (`../data/oro/falls/`) generada por `00_Preprocesamiento.ipynb`.
Esquema estandarizado de 14 columnas (8 canales inerciales tras Butterworth 4º @ 8 Hz):

`Dataset, Subject, Activity_Label, Activity_Code, Trial, Sample_Index, Ax, Ay, Az, Gx, Gy, Gz, AVM, GVM`

- **Activity_Label**: `Fall` o `ADL` (binario).
- **Activity_Code`: código crudo del dataset de origen (preserva trazabilidad con la taxonomía unificada F1..F10).
- **AVM / GVM**: magnitudes vectoriales de aceleración y giroscopio.

### Configuraciones Experimentales
| Config | Datasets       | Propósito                                          |
|--------|----------------|--------------------------------------------------------|
| set_a  | 5 datasets     | Incluir UMAFall (20→50 Hz + interpolación)               |
| set_b  | 4 datasets sin UMAFall | Evaluar impacto del resampleo sintético de UMAFall |


## 1 · Imports y constantes


In [ ]:
import os
import json
import pathlib
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import mixed_precision
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

# ---- Detección de GPU y configuración de runtime ------------------------------
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    mixed_precision.set_global_policy('mixed_float16')
    print(f"GPU(s) detectadas: {len(gpus)} | mixed_precision=ON (float16 en capas internas)")
else:
    print("Sin GPU detectada — fallback CPU. mixed_precision deshabilitado.")

FS = 50
WINDOW_SEC = 3.0
WINDOW_SIZE = int(FS * WINDOW_SEC)  # 150 timesteps
WINDOW_STEP = WINDOW_SIZE // 2      # 75 (50% overlap)
PEAK_MARGIN_SEC = 1.5
PEAK_MARGIN_SAMPLES = int(PEAK_MARGIN_SEC * FS)  # 75
N_FOLDS = 5

CHANNEL_COLS = ["Ax", "Ay", "Az", "Gx", "Gy", "Gz", "AVM", "GVM"]
N_CHANNELS = len(CHANNEL_COLS)  # 8

# ---- Hiperparámetros para GPU ------------------------------------------------
# Se usa el dataset completo (sin subsampleo). En CPU el entrenamiento será muy largo.
MAX_EPOCHS = 40
PATIENCE = 10
BATCH_SIZE = 512
LEARNING_RATE = 1e-3

ORO_DIR = pathlib.Path("../data/oro/falls")
MODELS_DIR = pathlib.Path("../data/modelos/falls")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

CONFIGS = {
    "set_a": ORO_DIR / "set_a.parquet",
    "set_b": ORO_DIR / "set_b.parquet",
}

print(f"FS={FS} Hz | WINDOW_SIZE={WINDOW_SIZE} | PEAK_MARGIN={PEAK_MARGIN_SEC}s | N_FOLDS={N_FOLDS}")
print(f"Channels ({N_CHANNELS}): {CHANNEL_COLS}")
print(f"epochs={MAX_EPOCHS} | batch={BATCH_SIZE} | patience={PATIENCE} | lr={LEARNING_RATE}")


## 2 · Carga local de Parquet


In [ ]:
def load_oro(path: pathlib.Path) -> pd.DataFrame:
    df = pd.read_parquet(path)
    print(f"  Cargado: {path.name} → {df.shape[0]:,} filas, datasets: {sorted(df['Dataset'].unique())}")
    return df


## 3 · Mapeo a la taxonomía unificada

Diccionario `(Dataset, Activity_Code) → grupo_unificado` derivado de
`doc/taxonomía-unificada.md`. La taxonomía preserva las etiquetas originales de
cada dataset y los agrupa por mecanismo biomecánico (F1–F8) o por cobertura parcial
(F9 = solo dirección, F10 = solo mecanismo de impacto).

## Taxonomía Unificada de Caídas

|  ID  | Grupo                                            | Descripción                                                                       |
| :--: | ------------------------------------------------ | --------------------------------------------------------------------------------- |
| **F1**  | **Resbalón al caminar**                          | Caídas producidas por pérdida de fricción durante la marcha.                      |
| **F2**  | **Tropiezo al caminar**                          | Caídas ocasionadas por un obstáculo durante la marcha.                            |
| **F3**  | **Tropiezo al trotar**                           | Tropiezo ocurrido durante jogging o carrera ligera.                               |
| **F4**  | **Síncope caminando o de pie**                   | Caídas provocadas por pérdida de conciencia desde una postura erguida.            |
| **F5**  | **Caída al intentar sentarse**                   | Pérdida del equilibrio durante la transición de pie a sentado.                    |
| **F6**  | **Caída al intentar levantarse**                 | Caídas durante la transición de sentado a de pie.                                 |
| **F7**  | **Síncope estando sentado**                      | Caídas por pérdida de conciencia desde una posición sentada.                      |
| **F8**  | **Caída desde la cama**                          | Caídas producidas durante movimientos o cambios de posición sobre la cama.        |
| **F9**  | **Caídas clasificadas únicamente por dirección** | El dataset únicamente informa la dirección de la caída, sin especificar su causa. |
| **F10** | **Caídas clasificadas por mecanismo de impacto** | El dataset diferencia únicamente la forma de impacto durante la caída.            |

In [ ]:
def build_taxonomy_map():
    """Mapeo (Dataset, Activity_Code) → grupo F1..F10.
    Fuentes: doc/taxonomía-unificada.md.
    Nota: A123–A126 de FallAllD son 'slipping' (no 'tripping'), por lo que
    caen en F1 (resbalón) junto con A103–A110.
    FallAllD en oro se almacena como int 101..135 (sin prefijo 'A'); KFall usa F01..F15."""
    m = {}
    for f, codes in {
        "F1": ["F01","F02","F03"], "F2": ["F04"], "F3": ["F05"], "F4": ["F06","F07"],
        "F5": ["F10","F11","F12"], "F6": ["F08","F09"], "F7": ["F13","F14","F15"],
    }.items():
        for c in codes: m[("SisFall", c)] = f
    for f, codes in {
        "F1": ["F13","F14","F15"], "F2": ["F11"], "F3": ["F12"], "F4": ["F09","F10"],
        "F5": ["F01","F02","F03"], "F6": ["F04","F05"], "F7": ["F06","F07","F08"],
    }.items():
        for c in codes: m[("KFall", c)] = f
    for f, codes in {
        "F1": ["103","104","105","106","107","108","109","110",
                "123","124","125","126"],
        "F2": ["101","102","121","122"],
        "F4": ["111","112","113","114","132","133","134","135"],
        "F5": ["115","116","117","118","119","120"],
        "F7": ["129","130","131"], "F8": ["127","128"],
    }.items():
        for c in codes: m[("FallAllD", c)] = f
    for f, codes in {"F5": ["5"], "F9": ["3","4"], "F10": ["1","2"]}.items():
        for c in codes: m[("UPFall", c)] = f
    for code in ["forwardFall", "backwardFall", "lateralFall"]:
        m[("UMAFall", code)] = "F9"
    return m

def map_to_unified_taxonomy(df):
    """Añade 'Fall_Type_Unified' preservando códigos originales.
    Asigna 'ADL' para actividades cotidianas y 'UNMAPPED' si el código de caída no está."""
    tax = build_taxonomy_map()
    out = df.copy()
    datasets = out["Dataset"].values
    codes = out["Activity_Code"].astype(str).values
    labels = out["Activity_Label"].values
    out["Fall_Type_Unified"] = [
        "ADL" if lbl == "ADL" else tax.get((ds, c), "UNMAPPED")
        for ds, c, lbl in zip(datasets, codes, labels)
    ]
    return out

TAXONOMY_ABBR = {
    "F1": "RC", "F2": "TC", "F3": "TT", "F4": "SC", "F5": "CS",
    "F6": "CL", "F7": "SS", "F8": "CC", "F9": "SD", "F10": "SM",
    "ADL": "ADL",
}
TAXONOMY_FULL = {
    "F1": "Resbalón caminando", "F2": "Tropiezo caminando",
    "F3": "Tropiezo trotando", "F4": "Síncope caminando/de pie",
    "F5": "Caída al intentar sentarse", "F6": "Caída al intentar levantarse",
    "F7": "Síncope estando sentado", "F8": "Caída desde la cama",
    "F9": "Solo dirección (UPFall/UMAFall)",
    "F10": "Solo mecanismo (UPFall)", "ADL": "Actividad de la Vida Diaria",
}


## 4 · Ventaneo precomputado

Se generan ventanas temporales de longitud fija a partir de las señales inerciales, utilizando un solapamiento del 50 %. Cada ventana se etiqueta como `ADL` o `Fall` según su contenido:

- Las ventanas de actividades cotidianas se etiquetan como `ADL`.
- En los trials de caída, se identifica el pico de `AVM`.
- Solo las ventanas cuyo centro se encuentra dentro de ±1.5 segundos del pico se etiquetan como `Fall`.
- Las ventanas restantes se conservan como `ADL`.

También se preparan las etiquetas de la taxonomía unificada y los identificadores de sujeto, que posteriormente se utilizan para la partición por grupos, el escalado y el entrenamiento del modelo.


In [ ]:
def build_group_id(df):
    df = df.copy()
    df["Group_ID"] = df["Dataset"] + "_" + df["Subject"].astype(str)
    return df

def create_windows(df, window_size=WINDOW_SIZE, window_step=WINDOW_STEP,
                   channel_cols=CHANNEL_COLS, peak_margin_samples=PEAK_MARGIN_SAMPLES):
    """Vectorizado. Etiqueta Fall=1 sólo a ventanas centradas ±peak_margin del pico AVM.
    Ventanas en trials ADL son siempre 0. Ventanas en trials Fall fuera de la ventana
    del pico son también 0 (caminar pre/post impacto, reposo)."""
    X_chunks, y_chunks, ft_chunks, gid_chunks = [], [], [], []
    df = build_group_id(df)
    half_w = window_size // 2
    for keys, group in df.groupby(["Dataset", "Subject", "Activity_Code", "Trial"], sort=False):
        group = group.sort_values("Sample_Index").reset_index(drop=True)
        n = len(group)
        if n < window_size:
            continue
        data = group[channel_cols].values
        is_fall = group["Activity_Label"].iloc[0] == "Fall"
        ft = group["Fall_Type_Unified"].iloc[0] if "Fall_Type_Unified" in group.columns else "ADL"
        gid = group["Group_ID"].iloc[0]
        n_w = (n - window_size) // window_step + 1
        starts = np.arange(n_w) * window_step
        idx = starts[:, None] + np.arange(window_size)[None, :]
        X_chunks.append(data[idx])

        if is_fall:
            peak_idx = int(np.argmax(group["AVM"].values))
            centers = starts + half_w
            labels = (np.abs(centers - peak_idx) <= peak_margin_samples).astype(np.int32)
        else:
            labels = np.zeros(n_w, dtype=np.int32)
        y_chunks.append(labels)
        ft_chunks.append(np.full(n_w, ft, dtype=object))
        gid_chunks.append(np.full(n_w, gid, dtype=object))
    X = np.concatenate(X_chunks, axis=0).astype(np.float32)
    y = np.concatenate(y_chunks)
    ft = np.concatenate(ft_chunks)
    gid = np.concatenate(gid_chunks)
    print(f"  Ventanas precomputadas: {len(X):,} (Fall={int(y.sum()):,}, ADL={int((y==0).sum()):,}, ratio Fall={y.mean():.3f})")
    return X, y, ft, gid

def fit_scaler(X_train):
    n, t, c = X_train.shape
    s = StandardScaler()
    s.fit(X_train.reshape(-1, c))
    return s

def apply_scaler(X, s):
    n, t, c = X.shape
    return s.transform(X.reshape(-1, c)).reshape(n, t, c).astype(np.float32)


## 5 · Función de pérdida y arquitectura CNN-BiLSTM

En esta sección se define el modelo CNN-BiLSTM para la clasificación binaria de ventanas temporales como `ADL` o `Fall`.

La red combina capas `Conv1D` para extraer patrones locales de las señales inerciales, normalización por lotes, activaciones `ReLU` y operaciones de *max pooling* para reducir la dimensión temporal. Posteriormente, una capa `Bidirectional LSTM` modela las dependencias temporales en ambas direcciones. Finalmente, `Dropout` ayuda a reducir el sobreajuste y una capa densa con activación sigmoide produce la probabilidad de caída.

El modelo se entrena mediante `BinaryCrossentropy`, el optimizador Adam y una tasa de aprendizaje de `1e-3`. Durante el entrenamiento se utiliza `class_weight={0: 2.0, 1: 1.0}` para ponderar las clases según la distribución esperada de los datos.

In [ ]:
def build_cnn_bilstm(window_size=WINDOW_SIZE, n_channels=N_CHANNELS):
    """Conv1D k=7 + k=5 + k=3 → campo receptivo ~52 timesteps (~1.04s @ 50 Hz)."""
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(window_size, n_channels)),
        tf.keras.layers.Conv1D(32, kernel_size=7, padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        tf.keras.layers.Conv1D(64, kernel_size=5, padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Conv1D(64, kernel_size=3, padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(1, activation="sigmoid", dtype="float32"),  # float32 obligatorio bajo mixed_float16
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


## 6 · Métricas binarias

Se utilizan tres métricas para evaluar la clasificación de ventanas como `Fall` o `ADL`:

- **Sensibilidad (Recall):** proporción de caídas correctamente detectadas.
  
  $$
  \text{Sensibilidad} = \frac{TP}{TP + FN}
  $$

- **Especificidad:** proporción de actividades cotidianas correctamente identificadas.
  
  $$
  \text{Especificidad} = \frac{TN}{TN + FP}
  $$

- **Precisión (Precision):** proporción de predicciones de caída que son correctas.
  
  $$
  \text{Precisión} = \frac{TP}{TP + FP}
  $$

Donde `TP` son verdaderos positivos, `TN` verdaderos negativos, `FP` falsos positivos y `FN` falsos negativos.

In [ ]:
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    return {
        "sensitivity": round(sens, 4),
        "specificity": round(spec, 4),
        "precision": round(prec, 4),
        "tp": tp, "fn": fn, "tn": tn, "fp": fp,
    }


## 7 · Orquestación del experimento

Se aplica validación cruzada estratificada por grupos mediante `StratifiedGroupKFold(K=5)`
sobre `Group_ID`, con una partición interna del 10 % mediante `GroupShuffleSplit`. El
`StandardScaler` se ajusta exclusivamente con los datos de entrenamiento de cada fold
para evitar filtraciones.

**GPU:** el modelo se entrena con el **dataset completo** (sin subsampleo) sobre
`BinaryCrossentropy`, `class_weight` dinámico según el ratio natural de cada fold, y
`EarlyStopping(patience=10)` + `ReduceLROnPlateau`. La capa densa final se fuerza a
`float32` para evitar `nan` con `mixed_precision=float16`. Como salvaguarda contra
*data mixing*, cada fold registra los datasets y las etiquetas presentes en el conjunto
de entrenamiento.


In [ ]:
def run_experiment(config_name, X, y, ft, gid, all_datasets):
    y_binary = y.astype(int)
    sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    folds = []

    for fold_idx, (tr, te) in enumerate(sgkf.split(np.zeros(len(X)), y_binary, gid)):
        print(f"\n--- Fold {fold_idx + 1}/{N_FOLDS} ---")
        train_idx, test_idx = tr, te

        gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=fold_idx)
        it, iv = next(gss.split(
            np.zeros(len(train_idx)),
            y_binary[train_idx],
            gid[train_idx],
        ))
        inner_train = train_idx[it]
        val_idx = train_idx[iv]

        train_datasets = sorted({gid[i].split("_")[0] for i in inner_train})
        train_labels = sorted({("Fall" if y[i] == 1 else "ADL") for i in inner_train})
        missing_ds = sorted(set(all_datasets) - set(train_datasets))
        missing_lb = sorted({"Fall", "ADL"} - set(train_labels))
        if missing_ds:
            print(f"  ⚠ Data mixing: train no contiene datasets: {missing_ds}")
        if missing_lb:
            print(f"  ⚠ Data mixing: train no contiene etiquetas: {missing_lb}")
        print(f"  train datasets={train_datasets}  labels={train_labels}")

        Xtr, ytr, fttr, gtr = X[inner_train], y[inner_train], ft[inner_train], gid[inner_train]
        Xva, yva, _, _       = X[val_idx],    y[val_idx],    ft[val_idx],    gid[val_idx]
        Xte, yte, ftte, _   = X[test_idx],   y[test_idx],   ft[test_idx],   gid[test_idx]
        print(f"  Train: {len(Xtr):,} (Fall={int(ytr.sum()):,})  Val: {len(Xva):,}  Test: {len(Xte):,} (Fall={int(yte.sum()):,})")

        scaler = fit_scaler(Xtr)
        Xtr = apply_scaler(Xtr, scaler)
        Xva = apply_scaler(Xva, scaler)
        Xte = apply_scaler(Xte, scaler)

        # class_weight dinámico según la razón inversa del batch de train (sin subsampleo)
        n_fall_tr = int(ytr.sum())
        n_adl_tr  = int((ytr == 0).sum())
        ratio = n_adl_tr / max(n_fall_tr, 1)
        class_weight = {0: 1.0, 1: ratio}
        print(f"  class_weight={class_weight}")

        model = build_cnn_bilstm()
        model.fit(
            Xtr, ytr,
            validation_data=(Xva, yva),
            epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, verbose=0,
            class_weight=class_weight,
            callbacks=[
                tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True),
                tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
            ],
        )

        y_pred = (model.predict(Xte, verbose=0).flatten() >= 0.5).astype(int)
        m = compute_metrics(yte, y_pred)
        folds.append({**m, "y_test": yte.tolist(), "y_pred": y_pred.tolist(),
                      "ft_test": ftte.tolist()})
        print(f"  Sens: {m['sensitivity']:.4f}  Spec: {m['specificity']:.4f}  Prec: {m['precision']:.4f}")

        del model, Xtr, Xva, Xte

        tf.keras.backend.clear_session()

    agg = {}
    for k in ("sensitivity", "specificity", "precision"):
        vals = [r[k] for r in folds]
        agg[f"{k}_mean"] = round(float(np.mean(vals)), 4)
        agg[f"{k}_std"] = round(float(np.std(vals)), 4)
    return {"config": config_name, "folds": folds, "summary": agg}


## 8 · Ejecución de ambos experimentos


In [ ]:
results = {}
for config_name, parquet_path in CONFIGS.items():
    print(f"\n{'='*60}\nCONFIGURACIÓN: {config_name}\n{'='*60}")
    df = load_oro(parquet_path)
    df = map_to_unified_taxonomy(df)

    unmapped = int(((df["Activity_Label"] == "Fall") & (df["Fall_Type_Unified"] == "UNMAPPED")).sum())
    if unmapped > 0:
        rows = df[
            (df["Activity_Label"] == "Fall") & (df["Fall_Type_Unified"] == "UNMAPPED")
        ][["Dataset", "Activity_Code"]].drop_duplicates()
        print(f"  ⚠ {unmapped:,} filas de caída sin mapeo ({len(rows)} códigos):")
        print(rows.to_string(index=False))

    n_subjects = df[["Dataset", "Subject"]].drop_duplicates().shape[0]
    print(f"  Sujetos: {n_subjects}")

    X, y, ft, gid = create_windows(df)
    all_datasets = sorted(df["Dataset"].unique())
    results[config_name] = run_experiment(config_name, X, y, ft, gid, all_datasets)
    del df, X, y, ft, gid


## 9 · Tabla comparativa y persistencia


In [ ]:
rows = []
for config_name, r in results.items():
    s = r["summary"]
    rows.append({
        "Config": config_name,
        "Sensibilidad": f"{s['sensitivity_mean']:.4f} ± {s['sensitivity_std']:.4f}",
        "Especificidad": f"{s['specificity_mean']:.4f} ± {s['specificity_std']:.4f}",
        "Precisión": f"{s['precision_mean']:.4f} ± {s['precision_std']:.4f}",
    })
print("Tabla Comparativa de Resultados (media ± desvío entre K folds)")
print("=" * 80)
print(pd.DataFrame(rows).set_index("Config").to_string())

out_path = MODELS_DIR / "comparison_results_gpu.json"
with open(out_path, "w") as f:
        json.dump(results, f, indent=2, default=str)
print(f"\n✓ Resultados persistidos en {out_path}")


In [ ]:
for config_name, parquet_path in CONFIGS.items():
    print(f"\nEntrenando modelo final para {config_name} (todo el dataset)...")
    df = load_oro(parquet_path)
    df = map_to_unified_taxonomy(df)

    X, y, ft, gid = create_windows(df)

    gss = GroupShuffleSplit(n_splits=1, test_size=0.05, random_state=42)
    tr_idx, va_idx = next(gss.split(np.zeros(len(X)), y, gid))

    scaler = fit_scaler(X[tr_idx])
    X_s = apply_scaler(X, scaler)

    n_fall_tr = int(y[tr_idx].sum())
    n_adl_tr  = int((y[tr_idx] == 0).sum())
    class_weight = {0: 1.0, 1: n_adl_tr / max(n_fall_tr, 1)}
    print(f"  class_weight={class_weight}")

    model = build_cnn_bilstm()
    model.fit(
        X_s[tr_idx], y[tr_idx],
        validation_data=(X_s[va_idx], y[va_idx]),
        epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, verbose=0,
        class_weight=class_weight,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
        ],
    )

    model_path = MODELS_DIR / f"{config_name}_final_gpu.keras"
    scaler_path = MODELS_DIR / f"{config_name}_scaler_gpu.joblib"
    model.save(model_path)
    joblib.dump(scaler, scaler_path)
    size_kb = model_path.stat().st_size / 1024
    print(f"  \u2713 Modelo: {model_path}  ({size_kb:.1f} KB)")
    print(f"  \u2713 Scaler: {scaler_path}")

    del df, X, model
    tf.keras.backend.clear_session()


## 10 · Matriz de confusión 2×2

Para cada modelo se agrega la matriz de confusión de los K folds. Dentro de cada cuadrante
se reportan los **3 grupos de la taxonomía unificada con mayor volumen de muestras**
(porcentaje respecto al total del cuadrante). Las etiquetas son abreviaturas de 2 letras;
la leyenda fuera del gráfico decodifica cada abreviatura.


In [ ]:
def top3_per_quadrant(y_test, y_pred, ft, quadrant_mask):
    if quadrant_mask.sum() == 0:
        return []
    groups = np.asarray(ft)[quadrant_mask]
    totals = pd.Series(groups).value_counts()
    top = totals.head(3)
    pct = (top / top.sum() * 100).round(1)
    return [(TAXONOMY_ABBR.get(g, g), f"{p:.1f}%") for g, p in pct.items()]

def plot_confusion_with_groups(config_name, fold_records):
    y_test = np.concatenate([np.asarray(f["y_test"]) for f in fold_records])
    y_pred = np.concatenate([np.asarray(f["y_pred"]) for f in fold_records])
    ft = np.concatenate([np.asarray(f["ft_test"]) for f in fold_records])

    tp = (y_test == 1) & (y_pred == 1)
    fn = (y_test == 1) & (y_pred == 0)
    tn = (y_test == 0) & (y_pred == 0)
    fp = (y_test == 0) & (y_pred == 1)

    quadrants = {"TP": tp, "FN": fn, "TN": tn, "FP": fp}
    quadrant_text = {}
    for name, mask in quadrants.items():
        items = top3_per_quadrant(y_test, y_pred, ft, mask)
        quadrant_text[name] = "\n".join(f"{a} {p}" for a, p in items) if items else "—"

    cm = np.array([[int(tn.sum()), int(fp.sum())], [int(fn.sum()), int(tp.sum())]])
    cm_pct = cm / max(cm.sum(), 1) * 100

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, cmap="Blues", vmin=0)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["Pred ADL", "Pred Fall"])
    ax.set_yticklabels(["Real ADL", "Real Fall"])

    cell_labels = {(0, 0): "TN", (0, 1): "FP", (1, 0): "FN", (1, 1): "TP"}
    for i in range(2):
        for j in range(2):
            quad = cell_labels[(i, j)]
            txt = f"{cm[i,j]:,}\n({cm_pct[i,j]:.1f}%)\n— top-3 —\n{quadrant_text[quad]}"
            ax.text(j, i, txt, ha="center", va="center", fontsize=9,
                    color="white" if cm[i,j] > cm.max()/2 else "black")

    sens = cm[1, 1] / (cm[1, 1] + cm[1, 0]) if (cm[1, 1] + cm[1, 0]) > 0 else 0
    spec = cm[0, 0] / (cm[0, 0] + cm[0, 1]) if (cm[0, 0] + cm[0, 1]) > 0 else 0
    ax.set_title(f"{config_name} — Matriz de confusión agregada (K folds)\n"
                 f"Sens: {sens:.1%}  ·  Spec: {spec:.1%}", fontweight="bold")
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.04)

    used = sorted({g for f in fold_records for g in np.asarray(f["ft_test"])})
    legend = "   ".join(f"{TAXONOMY_ABBR.get(g, g)}={TAXONOMY_FULL.get(g, g)}" for g in used)
    fig.text(0.5, -0.02, "Leyenda: " + legend, ha="center", fontsize=7, wrap=True)
    plt.tight_layout()
    return fig

for config_name, r in results.items():
    fig = plot_confusion_with_groups(config_name, r["folds"])
    plt.show()


## 11 · Conclusión

Este notebook (variante GPU) entrena ambos modelos sobre el **dataset completo**
sin subsampleo, usando `mixed_precision=float16` cuando hay GPU disponible y un
`class_weight` calculado dinámicamente a partir del ratio natural de cada fold.
Las métricas esperadas deberían igualar o superar las obtenidas en CPU gracias
al uso completo de las ~93k ventanas disponibles por configuración.